# logit-thermo 探索：3 つの谷を新しいものさし（かたまりで切り出す）で 25 本数え直す（CPU）

- **必要な secret：なし**。**ランタイム：CPU（標準）**。見込みは 10〜20 分。
- 探索（予想なし）。A2-2・A2-5・A2-6 の 25 本の生成（温度 0.6・0.7）で、探索 8.25 の新しいものさしで L・A・C を数え、割合・入り方・出方を
  `A2_explore/seg_univ_summary.json` に書きます（8.24 の数え直し）。**結果は表示しません。**
- 計算のコード：結晶の始まりの探索と A2-8 のコードは Drive の前のノートブックから取り出してハッシュで確かめ、`scripts/a2_explore_seg.py`・`a2_explore_seg_univ.py`（`c6faccb9191849b8…`）は埋め込み。
- **通知**：🚀・🎉・⚠️。名札は「logit-thermo 探索 3 つの谷を新しいものさしで 25 本（CPU）」。終わったら、またはエラーで止まったら、ランタイムを切ります。


In [ ]:
# セル1：通知、GPU、Drive
import os, time, urllib.request
NTFY_TOPIC = 'YOUR_NTFY_TOPIC'
NB_LABEL = 'logit-thermo 探索 3 つの谷を新しいものさしで 25 本（CPU）'
def notify(msg, tags='pig'):
    try:
        req = urllib.request.Request('https://ntfy.sh/' + NTFY_TOPIC, data=(msg + '\n[' + NB_LABEL + ']').encode('utf-8'),
                                     headers={'Title': 'logit-thermo - explore seg univ', 'Tags': tags})
        urllib.request.urlopen(req, timeout=10).read()
        print('通知を送りました：', msg)
    except Exception as e:
        print('通知は送れませんでした（実験には影響しません）：', type(e).__name__)
_NOTIFIED = {'error': False}
def _on_error(result):
    if result.error_in_exec is not None and not _NOTIFIED['error']:
        _NOTIFIED['error'] = True
        import traceback
        e = result.error_in_exec
        txt = ''.join(traceback.format_exception(type(e), e, e.__traceback__))
        v = os.environ.get('HF_TOKEN')
        if v: txt = txt.replace(v, '***')
        txt = txt.encode('utf-8')[-3300:].decode('utf-8', 'ignore')
        notify('⚠️ 止まりました（エラー）：\n' + txt, tags='warning')
if not globals().get('_HOOK_ON'):
    get_ipython().events.register('post_run_cell', _on_error); _HOOK_ON = True
notify('🚀 スタートしました（これが届けば通知の設定は OK です）', tags='rocket')

from google.colab import drive
drive.mount('/content/drive')


In [ ]:
# セル2：計算のコード
import math
import ast, glob, hashlib, json
def take_src(want):
    """want：{変数の名前: sha256}。Drive の logit-thermo のノートブックから、埋め込んだコードを取り出してハッシュで確かめる。"""
    got = {}
    for p in sorted(glob.glob('/content/drive/MyDrive/logit-thermo' + '/**/*.ipynb', recursive=True)):
        try:
            cells = json.load(open(p))['cells']
        except Exception:
            continue
        for c in cells:
            s = c['source'] if isinstance(c['source'], str) else ''.join(c['source'])
            if c.get('cell_type') != 'code' or 'SRC' not in s:
                continue
            try:
                body = ast.parse(s).body
            except SyntaxError:
                continue
            for n in body:
                if isinstance(n, ast.Assign) and isinstance(n.targets[0], ast.Name) and n.targets[0].id in want and n.targets[0].id not in got:
                    v = ast.literal_eval(n.value)
                    if hashlib.sha256(v.encode()).hexdigest() == want[n.targets[0].id]:
                        got[n.targets[0].id] = v
        if len(got) == len(want):
            break
    miss = [k for k in want if k not in got]
    assert not miss, ('Drive のノートブックからコードを取り出せませんでした（前のノートブックを動かしていませんか）：', miss)
    return got
_s = take_src({'SRCON': '0b39f997b72f33d3903df735b9e6525b9786cc714e4457e4442a2d6db1c803e9', 'SRC8': '06a93c4eec9f3e770ca9e2f3f791a94b37153c908becd0f83d825acfbd11bacd'})
SRCON, SRC8 = _s['SRCON'], _s['SRC8']
SRCSG = '"""探索 8.25：窓の形の見かけを除いた 3 つの状態のものさし（かたまりで切り出す）で、8.22 をやり直す（2026-10-30 18:2x、CPU、判定しない）。\nscripts/a2_explore_onset.py（copy_marks）・scripts/a2_8_vh.py（phases）・scripts/a2_explore_three.py（rates・vh_step）を実行したのと同じ名前空間で実行する。\n\n8.24 で分かったこと：8.21 の窓（64 歩、16 歩おき）のラベルは、周期 17 歩以上のきれいな結晶がいきなり終わっても、\n溶け方を「C → A → L」に見せてしまう（窓の中で結晶の部分が減ると、周期の強さ P が写しの割合 r より先に 0.5 を切るため）。\n新しいものさし：\n1) 相（液体・結晶）は A2-8 と同じ（写しの印、中心の窓 32、入口 0.5・出口 0.25 の履歴つき）。歩ごと（歩 16〜495）。\n2) 結晶の歩の「形」は、写しの印が付いた歩それぞれの「写しのずれ」（その 4 つ組が直前に出た位置までの歩数）で決める。\n   歩 t のずれ ＝ t を中心にした 33 歩の中の、写しの印が付いた歩のずれの下から 10% の値（印がなければ直前のまま）。\n   中央値にしないのは、結晶に傷（ほかのトークン）があると、傷のあとの 4 つ組は 1 周期前ではなく 2 周期前と合い、ずれが周期の 2 倍に出るため\n   （作り物で、周期 23・傷 10% のとき、中央値では C ⇄ A のちらつきが 100 回出たが、下から 10% では 8 回）。\n   ずれ ≤ 32 → C（短い周期）、ずれ > 32 → A（長い周期・周期なし）。\n   ずれはトークンそのものの性質なので、結晶の終わりで窓の中の結晶が減っても、ずれは変わらない（見かけが出にくい）。\n速さは 1 歩あたり。段ごとの平衡定数・ファントホッフは 8.22 と同じ（rates・vh_step、両向きそれぞれ 8 回以上の温度）。\n"""\nimport numpy as np\n\nSG_HALF = 16          # ずれの中央値をとる幅（t−16〜t+16）\nSG_SHORT = 32         # これ以下のずれを「短い周期」\nSG_Q = 10             # ずれの下から何 % の値を使うか\n\n\ndef copy_lags(seq):\n    """→ lag（長さ L、整数。写しの印がない歩は 0）。t 歩目で終わる 4 つ組が、直前に出た位置（終わりの歩）までの歩数。"""\n    s = [int(x) for x in seq]\n    L = len(s)\n    lag = np.zeros(L, np.int64)\n    last = {}\n    for t in range(3, L):\n        g = (s[t - 3], s[t - 2], s[t - 1], s[t])\n        if g in last:\n            lag[t] = t - last[g]\n        last[g] = t\n    return lag\n\n\ndef seg_labels(seq):\n    """→ 歩 16〜495 のラベルの列（\'L\'・\'A\'・\'C\'）。"""\n    c = copy_marks(seq)\n    st, t = phases(c)\n    lag = copy_lags(seq)\n    out, prev = [], \'A\'\n    for ph, tt in zip(st, t):\n        if ph == 0:\n            out.append(\'L\'); prev = \'A\'\n            continue\n        a, b = max(0, tt - SG_HALF), min(len(lag), tt + SG_HALF + 1)\n        w = lag[a:b]; w = w[w > 0]\n        if len(w):\n            prev = \'C\' if np.percentile(w, SG_Q) <= SG_SHORT else \'A\'\n        out.append(prev)\n    return out\n\n\ndef seg_counts(seq):\n    labs = seg_labels(seq)\n    occ = {s: 0 for s in \'LAC\'}; tr = {}\n    for x, y in zip(labs[:-1], labs[1:]):\n        occ[x] += 1\n        if x != y:\n            tr[x + y] = tr.get(x + y, 0) + 1\n    return occ, tr, labs\n\n\ndef seg_counts_by_doc(toks):\n    """toks：(nT, S, nD, L) → (nT, nD, 9)：[L, A, C にいた歩の数, LA, AL, AC, CA, LC, CL の回数]（8.22 の counts_by_doc と同じ並び）。"""\n    nT, S, nD, _ = toks.shape\n    keys = [\'LA\', \'AL\', \'AC\', \'CA\', \'LC\', \'CL\']\n    out = np.zeros((nT, nD, 9))\n    for k in range(nT):\n        for j in range(nD):\n            for s in range(S):\n                occ, tr, _ = seg_counts(toks[k, s, j])\n                out[k, j, :3] += [occ[\'L\'], occ[\'A\'], occ[\'C\']]\n                out[k, j, 3:] += [tr.get(x, 0) for x in keys]\n    return out\n\n\ndef after_CA(toks, ks):\n    """C → A のあと、A がどこへ行くか（C・L・終わり）。"""\n    res = {\'C\': 0, \'L\': 0, \'end\': 0}\n    for k in ks:\n        for s in range(toks.shape[1]):\n            for j in range(toks.shape[2]):\n                labs = seg_labels(toks[k, s, j])\n                for i in range(len(labs) - 1):\n                    if labs[i] == \'C\' and labs[i + 1] == \'A\':\n                        m = i + 1\n                        while m < len(labs) and labs[m] == \'A\':\n                            m += 1\n                        res[labs[m] if m < len(labs) else \'end\'] += 1\n    return res\n\n\ndef self_test_seg(seed=1):\n    """作り物：きれいな結晶をいきなり入れて、いきなり終わらせる。短い周期は L → C → L、長い周期は L → A → L、\n    途中で周期が 61 から 7 に縮むものは L → A → C → L とだけ数えること。"""\n    rng = np.random.default_rng(seed)\n    def make(parts):\n        s = list(rng.integers(100, 30000, 150))\n        for per, n in parts:\n            s += list(np.tile(rng.integers(100, 30000, per), 600)[:n])\n        s += list(rng.integers(100, 30000, 512 - len(s)))\n        return np.array(s[:512])\n    out = {}\n    for name, parts in {\'p3\': [(3, 180)], \'p7\': [(7, 180)], \'p23\': [(23, 200)], \'p29\': [(29, 200)],\n                        \'p45\': [(45, 220)], \'p61\': [(61, 240)], \'p61to7\': [(61, 200), (7, 120)]}.items():\n        tot = {}\n        for _ in range(20):\n            _, tr, _ = seg_counts(make(parts))\n            for k, v in tr.items():\n                tot[k] = tot.get(k, 0) + v\n        out[name] = tot\n    for n in (\'p3\', \'p7\', \'p23\', \'p29\'):\n        assert out[n].get(\'LC\', 0) == 20 and out[n].get(\'CL\', 0) == 20 and sum(out[n].values()) == 40, (n, out[n])\n    for n in (\'p45\', \'p61\'):\n        assert out[n].get(\'LA\', 0) == 20 and out[n].get(\'AL\', 0) == 20 and sum(out[n].values()) == 40, (n, out[n])\n    o = out[\'p61to7\']\n    assert o.get(\'LA\', 0) == 20 and o.get(\'AC\', 0) == 20 and o.get(\'CL\', 0) == 20 and sum(o.values()) == 60, o\n    # 傷 10% の結晶（周期 23）：ちらつき（C ⇄ A）が、入って出る回数の 2 割より少ないこと\n    tot = {}\n    for _ in range(30):\n        s = list(rng.integers(100, 30000, 150)); body = np.tile(rng.integers(100, 30000, 23), 600)[:220].copy()\n        m = rng.random(220) < 0.10; body[m] = rng.integers(100, 30000, m.sum())\n        s += list(body); s += list(rng.integers(100, 30000, 512 - len(s)))\n        _, tr, _ = seg_counts(np.array(s))\n        for k, v in tr.items():\n            tot[k] = tot.get(k, 0) + v\n    out[\'p23_defect10\'] = tot\n    assert tot.get(\'CA\', 0) + tot.get(\'AC\', 0) < 0.2 * (tot.get(\'LC\', 0) + tot.get(\'CL\', 0)), tot\n    return out\n'
SRCSU = '"""探索 8.29：8.24（3 つの谷は 25 本でも同じか）を、8.25 の新しいものさし（かたまりで切り出す）でやり直す（2026-10-31 09:3x、CPU、判定しない）。\nscripts/a2_explore_onset.py（copy_marks）・scripts/a2_8_vh.py の phases・scripts/a2_explore_seg.py を実行したのと同じ名前空間で実行する。\n材料：A2-2・A2-5・A2-6 の 25 本（温度 0.6〜1.5 の 15 個、サンプル 2 × 記事 40）。本ごとに：温度 0.6・0.7 の L・A・C の歩の割合、\n温度 0.6・0.7 を足した移り変わりの回数、C から出るうちの L への割合、C に入るうちの L からの割合、C → A のあとの行き先。\n"""\nimport numpy as np\n\nSU_RUNS = {\n    \'A2_2\': [\'pythia-1b\', \'qwen2.5-0.5b\', \'qwen2.5-0.5b-inst\', \'qwen2.5-1.5b\', \'olmo2-1b\', \'smollm2-1.7b\', \'smollm2-1.7b-inst\'],\n    \'A2_5\': [\'pythia-70m\', \'pythia-160m\', \'pythia-410m\', \'pythia-1b\', \'pythia-1.4b\', \'pythia-2.8b\',\n             \'pythia-1b@step8000\', \'pythia-1b@step24000\', \'pythia-1b@step64000\'],\n    \'A2_6\': [\'qwen2.5-0.5b\', \'qwen2.5-1.5b\', \'qwen2.5-3b\', \'smollm2-135m\', \'smollm2-360m\', \'smollm2-1.7b\',\n             \'qwen3-0.6b\', \'qwen3-1.7b\', \'qwen3-4b\'],\n}\nSU_TEMPS = [0.6, 0.7, 0.8, 0.85, 0.9, 0.95, 1.0, 1.05, 1.1, 1.15, 1.2, 1.25, 1.3, 1.4, 1.5]\n\n\ndef one_run_seg(toks):\n    keys = [\'LA\', \'AL\', \'AC\', \'CA\', \'LC\', \'CL\']\n    row, tr = {}, {k: 0 for k in keys}\n    after = {\'C\': 0, \'L\': 0, \'end\': 0}\n    for T in (0.6, 0.7):\n        k = SU_TEMPS.index(T)\n        occ = np.zeros(3)\n        for s in range(toks.shape[1]):\n            for j in range(toks.shape[2]):\n                o, t, labs = seg_counts(toks[k, s, j])\n                occ += [o[\'L\'], o[\'A\'], o[\'C\']]\n                for x in keys:\n                    tr[x] += t.get(x, 0)\n                for i in range(len(labs) - 1):\n                    if labs[i] == \'C\' and labs[i + 1] == \'A\':\n                        m = i + 1\n                        while m < len(labs) and labs[m] == \'A\':\n                            m += 1\n                        after[labs[m] if m < len(labs) else \'end\'] += 1\n        occ = occ / occ.sum()\n        row[f\'frac_{T}\'] = dict(L=float(occ[0]), A=float(occ[1]), C=float(occ[2]))\n    row[\'trans_0.6_0.7\'] = tr\n    row[\'C_exit_to_L\'] = tr[\'CL\'] / (tr[\'CL\'] + tr[\'CA\']) if (tr[\'CL\'] + tr[\'CA\']) else None\n    row[\'C_entry_from_L\'] = tr[\'LC\'] / (tr[\'LC\'] + tr[\'AC\']) if (tr[\'LC\'] + tr[\'AC\']) else None\n    row[\'after_CA\'] = after\n    return row\n\n\ndef seg_univ(base):\n    res = {}\n    for folder, names in SU_RUNS.items():\n        for n in names:\n            Z = np.load(f\'{base}/{folder}/{n}.npz\')\n            assert [round(float(t), 4) for t in Z[\'temps\']] == SU_TEMPS\n            res[f\'{folder}/{n}\'] = one_run_seg(Z[\'toks\'])\n    return res\n\n\ndef self_test_su():\n    rng = np.random.default_rng(0)\n    toks = rng.integers(100, 30000, (15, 2, 3, 512))\n    toks[0, 0, 0, 150:330] = np.tile([5, 6, 7, 8, 9, 10, 11], 60)[:180]\n    r = one_run_seg(toks)\n    assert r[\'trans_0.6_0.7\'][\'LC\'] == 1 and r[\'trans_0.6_0.7\'][\'CL\'] == 1 and r[\'C_exit_to_L\'] == 1.0\n    return \'OK\'\n'
assert hashlib.sha256(SRCSG.encode()).hexdigest() == 'dd8e4a81150beb02b0aa61d65add91b179fdfda4706c6e4423b12ec59f0e1d04' and hashlib.sha256(SRCSU.encode()).hexdigest() == 'c6faccb9191849b870d4f9a81a688d6ac40a18c8da7661356e55f0935dececc7'
exec(SRCON)
exec(SRC8)
exec(SRCSG)
exec(SRCSU)
def _clean(o):
    if isinstance(o, dict): return {str(k): _clean(v) for k, v in o.items()}
    if isinstance(o, (list, tuple)): return [_clean(v) for v in o]
    if isinstance(o, (np.floating, np.integer)): o = o.item()
    if isinstance(o, float) and not math.isfinite(o): return None
    return o
print('自己テスト：', self_test_onset(), self_test_seg() is not None, self_test_su())


In [ ]:
# セル3：数えて保存（結果は表示しない）
try:
    r = seg_univ('/content/drive/MyDrive/logit-thermo')
    r = dict(runs=r, onset_sha256='0b39f997b72f33d3903df735b9e6525b9786cc714e4457e4442a2d6db1c803e9', a2_8_sha256='06a93c4eec9f3e770ca9e2f3f791a94b37153c908becd0f83d825acfbd11bacd', seg_sha256='dd8e4a81150beb02b0aa61d65add91b179fdfda4706c6e4423b12ec59f0e1d04', seg_univ_sha256='c6faccb9191849b870d4f9a81a688d6ac40a18c8da7661356e55f0935dececc7')
    json.dump(_clean(r), open('/content/drive/MyDrive/logit-thermo' + '/A2_explore/seg_univ_summary.json', 'w'), ensure_ascii=False, separators=(',', ':'))
    print('保存しました')
except Exception as e:
    import traceback
    _NOTIFIED['error'] = True
    txt = ''.join(traceback.format_exception(type(e), e, e.__traceback__))
    v = os.environ.get('HF_TOKEN')
    if v: txt = txt.replace(v, '***')
    txt = txt.encode('utf-8')[-3300:].decode('utf-8', 'ignore')
    notify('⚠️ 探索 3 つの谷を新しいものさしで 25 本で止まりました。ランタイムを切ります：\n' + txt, tags='warning')
    if globals().get('AUTO_DISCONNECT', True):
        from google.colab import drive as _d, runtime as _r
        _d.flush_and_unmount(); time.sleep(3); _r.unassign()
    raise


In [ ]:
# 最後：終わりの通知と、ランタイムを切る
notify('🎉 終わりました：新しいものさしの 25 本の要約を A2_explore/seg_univ_summary.json に保存しました', tags='tada')
if globals().get('AUTO_DISCONNECT', True):
    from google.colab import drive as _drive, runtime as _runtime
    _drive.flush_and_unmount(); time.sleep(3); _runtime.unassign()
